# 07 — Extracting 10-K items

**Purpose.** This notebook walks the filing index and pulls named items out of
each annual report through the sec-api.io Extractor, writing one text file per
filing-item and one status line per attempt.

**One run, one tree.** Every parameter comes from the environment, and the run's
name decides where everything lands, so two runs never share an output directory
or a status log:

| Variable | Default | Meaning |
|---|---|---|
| `RUN_NAME` | `items` | Output tree under `interim/sec/` |
| `ITEMS` | `1,1A,1B,1C,7,7A` | Items to extract, comma separated |
| `SP500_ONLY` | `0` | Restrict to firms ever in the S&P 500 |
| `SHARD`, `N_SHARDS` | `0`, `1` | Which slice of the index this task takes |
| `N_CONCURRENT` | `N_SHARDS` | How many tasks actually run at once |
| `TARGET_RATE` | `7.0` | **Aggregate** requests/second across the running tasks |

**`TARGET_RATE` is the parameter that matters, and it is aggregate on purpose.**
The sec-api rate limit is account-wide, not per connection: measured against the
live API, goodput rises to about 8.8 requests/second and then *falls*, because
past the ceiling the API returns HTTP 429 instead of data. Shards therefore do
not multiply throughput — they divide it. Each task paces itself at
`TARGET_RATE / N_CONCURRENT`, so the array offers the same total rate whether it
runs as 4 tasks or 40, and adding tasks buys resilience rather than speed.

`N_CONCURRENT` is separate from `N_SHARDS` because SLURM's `--array=0-63%12`
means 64 shards of which only **12** run at a time. Pacing against 64 would run
the array at an eighth of the intended rate for no reason; pacing against 12 is
correct. The job script passes the `%N` throttle through as `N_CONCURRENT`, and
it defaults to `N_SHARDS` for the unthrottled case.

The first attempt at this notebook got that backwards: 4 tasks each pacing at 8
requests/second offered 32, roughly four times the ceiling, so most calls were
rejected. The client then treated each 429 as a transient failure and backed off
exponentially — 1, 2, 4, 8, 16 seconds — against a limit whose real penalty is
the remainder of the current second. Aggregate throughput collapsed to about 4
requests/second, an eighth of what a single thread sustains. `secapi.py` now
pauses a flat 0.25 s on 429 and retries without spending a retry, and the rate is
declared once, here, as a total.

**Status values.** Every attempt is recorded, and every value except `error` is
final — which is what makes a resume pick up exactly the transient failures and
nothing else.

| Status | Meaning |
|---|---|
| `ok` | Section written |
| `skipped` | Already on disk from an earlier run |
| `empty` | The API returned nothing — the item is not in this filing |
| `unsupported` | The Extractor cannot split this filing at all |
| `error` | A transient failure that outlived the retry budget — retried on resume |

**On `empty`.** These were audited against the filings themselves rather than
assumed: for 313 empty results the item heading was searched for in the archived
document. 311 were genuine absences that line up exactly with the disclosure
rules — Item 1A and 1B required from December 2005, Item 1C from December 2023 —
and the heading appears nowhere in the filing. Two were extraction failures, one
of them a real loss of 26,374 characters of MD&A whose heading sat inside a table
cell. So `empty` is overwhelmingly meaningful, but it is not a guarantee, and the
verification below reports the empty rate by era rather than driving it to zero.

One trap worth stating because it is invisible in the data: of the pre-2005
filings with an empty Item 1A, roughly two thirds *do* discuss risk factors,
under unnumbered headings like "Factors That May Affect Future Results". "Item 1A
is empty" is a correct statement about the numbered item and a false one about
the filing's content. Any pre/post-2005 comparison of risk-factor text will show
a jump at 2005 that is a reporting-format artifact.

**Outputs.**

| File | Path |
|---|---|
| Item text | `interim/sec/{RUN_NAME}/item{X}/{cik}/{accessionNo}.txt` |
| Status log | `interim/sec/{RUN_NAME}/status/extract_{shard}.jsonl` |
| Consolidated | `interim/sec/{RUN_NAME}/status/extract_status.parquet` |

## Setup

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, "../lib")

import pandas as pd

from paths import SEC_RAW, sec_run_dir
from secapi import SecApiClient, SecApiError, UnsupportedFiling, load_api_key

RUN_NAME = os.environ.get("RUN_NAME", "items")
ITEMS = tuple(i.strip() for i in os.environ.get("ITEMS", "1,1A,1B,1C,7,7A").split(",") if i.strip())
SP500_ONLY = os.environ.get("SP500_ONLY", "0") == "1"
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 1))
# Tasks actually running at once, which is what shares the account-wide limit —
# not the shard count. With `--array=0-63%12` these are 64 and 12.
N_CONCURRENT = int(os.environ.get("N_CONCURRENT", N_SHARDS))
TARGET_RATE = float(os.environ.get("TARGET_RATE", 7.0))

RUN_DIR = sec_run_dir(RUN_NAME)
STATUS_DIR = RUN_DIR / "status"

filings = pd.read_parquet(SEC_RAW / "filings.parquet")
if SP500_ONLY:
    filings = filings.loc[filings["in_sp500"]]

work = filings.iloc[SHARD::N_SHARDS].reset_index(drop=True)

# Aggregate, divided by the number of tasks sharing the account-wide limit.
shard_rate = TARGET_RATE / N_CONCURRENT
client = SecApiClient(api_key=load_api_key(env_file=".env"),
                      rate_per_sec=shard_rate, timeout=180)

print(f"run        : {RUN_NAME}  ->  {RUN_DIR}")
print(f"items      : {', '.join(ITEMS)}")
print(f"scope      : {'S&P 500 only' if SP500_ONLY else 'full sample'}")
print(f"shard      : {SHARD} of {N_SHARDS}  ({N_CONCURRENT} running at once)")
print(f"rate       : {TARGET_RATE} req/s aggregate -> {shard_rate:.3f} req/s this task")
print(f"filings    : {len(filings):,} in scope, {len(work):,} in this shard")
print(f"calls      : {len(work) * len(ITEMS):,} in this shard, "
      f"{len(filings) * len(ITEMS):,} in the run")
print(f"projected  : {len(filings) * len(ITEMS) / TARGET_RATE / 3600:.1f} h at {TARGET_RATE} req/s")

## Step 1 — One filing-item

The API is asked for the item and the answer is written only when it is
non-empty, so an empty file never appears on disk and `skipped` always means real
content. The write goes to a temporary name first: a task killed at the wall
clock cannot leave a half-written section that the next run would accept.

In [ ]:
def extract_one(row, item: str) -> dict:
    """Extract one item from one filing, returning what happened."""
    target = RUN_DIR / f"item{item}" / row.cik_unpadded / f"{row.accessionNo}.txt"
    record = {
        "accessionNo": row.accessionNo,
        "cik": row.cik_unpadded,
        "gvkey": row.gvkey,
        "fyear": int(row.fyear),
        "formType": row.formType,
        "item": item,
        "path": str(target),
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if target.exists():
        return {**record, "status": "skipped", "chars": target.stat().st_size}

    try:
        text = client.get_section(row.filingUrl, item)
    except UnsupportedFiling as exc:
        # Terminal. Filed under "error" it would be re-attempted on every resume
        # and would fail the run's own completeness check forever.
        return {**record, "status": "unsupported", "chars": 0, "message": str(exc)[:300]}
    except SecApiError as exc:
        return {**record, "status": "error", "chars": 0, "message": str(exc)[:300]}

    if not text or not text.strip():
        return {**record, "status": "empty", "chars": 0}

    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(".txt.partial")
    tmp.write_text(text)
    tmp.replace(target)
    return {**record, "status": "ok", "chars": len(text)}

## Step 2 — Walk the shard

Status lines are appended and flushed as they happen, so a task killed at the
wall clock loses nothing but the call in flight. The resume set is built from
**every** shard's log in this run, not just this task's: shards partition the
index with `iloc[SHARD::N_SHARDS]` and never overlap, so the union is safe, and
it is what lets a differently-sharded re-run skip work already done instead of
repeating it.

In [ ]:
# Every status except "error" is final.
TERMINAL = {"ok", "skipped", "empty", "unsupported"}

status_path = STATUS_DIR / f"extract_{SHARD:03d}.jsonl"

done = set()
for log in sorted(STATUS_DIR.glob("extract_*.jsonl")):
    for line in log.read_text().splitlines():
        if line.strip():
            row = json.loads(line)
            if row["status"] in TERMINAL:
                done.add((row["accessionNo"], row["item"]))

counts = {"ok": 0, "skipped": 0, "empty": 0, "unsupported": 0, "error": 0}
todo = [(r, item) for r in work.itertuples() for item in ITEMS
        if (r.accessionNo, item) not in done]
print(f"{len(done):,} filing-items already recorded, {len(todo):,} still to fetch")

started = datetime.now(timezone.utc)
with status_path.open("a") as fh:
    for n, (row, item) in enumerate(todo, 1):
        record = extract_one(row, item)
        fh.write(json.dumps(record) + "\n")
        fh.flush()
        counts[record["status"]] += 1
        if n % 1000 == 0:
            secs = (datetime.now(timezone.utc) - started).total_seconds()
            rate = n / secs if secs else 0
            eta = (len(todo) - n) / rate / 3600 if rate else 0
            print(f"  {n:,}/{len(todo):,}  {rate:.2f} req/s  eta {eta:.1f} h  "
                  + " ".join(f"{k}={v:,}" for k, v in counts.items() if v))

secs = (datetime.now(timezone.utc) - started).total_seconds()
print()
print("totals:", " ".join(f"{k}={v:,}" for k, v in counts.items()))
print(f"{len(todo):,} attempts in {secs / 60:.1f} min "
      f"= {len(todo) / secs if secs else 0:.2f} req/s for this task")
print(f"{client.n_requests:,} requests, {client.n_retries:,} retries, "
      f"{client.n_rate_limited:,} rate-limit pauses")

## Step 3 — Consolidate the status log

**Only an unsharded pass consolidates.** Under `--array` every task runs this
same notebook and finishes at a different time; if each wrote the shared parquet
they would overwrite one another mid-write, and the survivor would be whichever
task happened to finish last. A sharded run therefore reports its own numbers and
writes nothing shared. Re-running the notebook once with `N_SHARDS=1` after the
array completes produces the consolidated file, and costs nothing: every
filing-item is already recorded, so Step 2 finds an empty to-do list.

In [ ]:
IS_AGGREGATE = N_SHARDS == 1

if IS_AGGREGATE:
    frames = [pd.read_json(p, lines=True) for p in sorted(STATUS_DIR.glob("extract_*.jsonl"))]
    status = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    status.to_parquet(STATUS_DIR / "extract_status.parquet", index=False)
    print(f"consolidated {len(frames)} shard logs -> {STATUS_DIR / 'extract_status.parquet'}")
else:
    status = pd.read_json(status_path, lines=True) if status_path.exists() else pd.DataFrame()
    print(f"shard {SHARD}: reporting on this shard's own log only; "
          "re-run with N_SHARDS=1 to consolidate")
print()

assert len(status), "no status rows at all — Step 2 wrote nothing"
# A filing-item can appear twice across re-runs; the final state is what counts.
status = status.drop_duplicates(["accessionNo", "item"], keep="last")

pivot = status.pivot_table(index="item", columns="status", values="accessionNo",
                           aggfunc="count", fill_value=0)
print(pivot.to_string())
print()
got = status.loc[status["status"].isin(["ok", "skipped"])]
if len(got):
    print(f"characters extracted: {got['chars'].sum() / 1e9:.3f} billion")
    print(got.groupby("item")["chars"].describe()[["count", "mean", "50%", "max"]].to_string())

## Verification

Five checks. The first two are stop conditions; the rest describe the corpus.
Under `--array` they run against the shard's own log, so each task verifies its
own work; the consolidation pass then runs them over the whole run.

In [ ]:
errors = status.loc[status["status"] == "error"]
assert len(errors) == 0, (
    f"{len(errors):,} extractions failed after retries — re-run to resume. "
    f"first: {errors.iloc[0]['accessionNo']} / item {errors.iloc[0]['item']}"
)
print("1) no unresolved transient errors")

claimed = status.loc[status["status"] == "ok", "path"]
missing = [p for p in claimed if not Path(p).exists()]
assert not missing, f"{len(missing):,} files recorded as written are not on disk"
print(f"2) all {len(claimed):,} written files are on disk")

covered = status.drop_duplicates(["accessionNo", "item"])
expected = len(work) * len(ITEMS) if not IS_AGGREGATE else len(filings) * len(ITEMS)
print(f"3) {len(covered):,} of {expected:,} filing-items attempted "
      f"({len(covered) / expected:.1%})")

era = status.assign(era=pd.cut(status["fyear"], [0, 1996, 2004, 2022, 2100],
                               labels=["<1997", "1997-2004", "2005-2022", "2023+"]))
rate = era.assign(is_empty=era["status"].eq("empty")).pivot_table(
    index="era", columns="item", values="is_empty", aggfunc="mean", observed=True)
print("4) share of extractions returning nothing, by disclosure era:")
print(rate.map("{:.1%}".format).to_string())
print("   Eras are cut at the rule changes: 7A from 1997, 1A and 1B from")
print("   December 2005, 1C from December 2023. A high empty rate before an")
print("   item's own cut is the regime, not a bug.")

panel = got.drop_duplicates(["gvkey", "fyear", "item"])
print(f"\n5) firm-year coverage: {panel['gvkey'].nunique():,} firms, "
      f"{len(panel.drop_duplicates(['gvkey', 'fyear'])):,} firm-years")

uns = status.loc[status["status"] == "unsupported"].drop_duplicates("accessionNo")
if len(uns):
    print(f"\n   {len(uns):,} filings the Extractor could not split at all. Every "
          "bare-CIK URL was repaired in 06, so anything here is a new case and "
          "is worth looking at rather than accepting.")
    print(uns.groupby("fyear").size().to_string())
else:
    print("\n   no unsupported filings — the URL repair in 06 held")